# Inspect and prepare HEST-1k human Visium inputs

This notebook documents the data objects consumed by HistoOmniST without
wrapping preparation scripts in shell commands. Public metadata, fixed
split manifests and gene lists are included in the repository. Raw HEST
images and count matrices must be obtained from HEST and remain outside
Git under `data/HEST-1k/raw/` and `data/HEST-1k/processed/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the HistoOmniST repository.")


ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

from histoomnist.data.size_factor import compute_size_factor, log_size_factor
from histoomnist.data.spot_table import load_spot_table
from histoomnist.eval.leakage_checks import assert_disjoint_groups


## 1. Restrict HEST metadata to human Visium sections


In [ ]:
metadata = pd.read_csv(ROOT / "data/HEST-1k/HEST_v1_3_0.csv")
human_visium = metadata.loc[
    metadata["species"].eq("Homo sapiens")
    & metadata["st_technology"].eq("Visium")
].copy()

print(f"All HEST rows: {len(metadata):,}")
print(f"Human Visium rows: {len(human_visium):,}")
display(human_visium[["id", "organ", "disease_state", "species", "st_technology"]].head())
display(human_visium["organ"].value_counts().rename_axis("organ").to_frame("slides"))


## 2. Inspect the processed-data contract and fixed slide split

One manifest row represents one slide. Feature, count, coordinate, SF,
spot and gene files are aligned before a slide is marked usable. The
headline partition is assigned at slide level, so spots from one slide
can never appear in more than one split.


In [ ]:
manifest_path = ROOT / "data/HEST-1k/manifests/human_visium_sf_manifest_context.csv"
split_path = ROOT / "data/HEST-1k/splits/leave_slide_out.csv"
manifest = pd.read_csv(manifest_path)
split_manifest = pd.read_csv(split_path)

assert_disjoint_groups(split_manifest, ["sample_id"])
assert split_manifest.groupby("sample_id")["split"].nunique().max() == 1

display(manifest.head(3))
display(split_manifest["split"].value_counts().rename_axis("split").to_frame("slides"))
print("No slide appears in more than one train/validation/test split.")


## 3. Validate one locally prepared slide

The following cell automatically selects the first manifest row whose
required processed arrays are present. If no raw assets have been
downloaded yet, it prints the exact expected files without failing the
notebook.


In [ ]:
base_dir = manifest_path.parent
required_columns = ["features_path", "counts_path", "coords_path", "size_factor_path"]

def row_is_available(row: pd.Series) -> bool:
    return all((base_dir / str(row[column])).resolve().exists() for column in required_columns)

available_rows = manifest.loc[manifest.apply(row_is_available, axis=1)]
if available_rows.empty:
    example_row = manifest.iloc[0]
    print("No processed HEST slide is present in this checkout.")
    print("Expected files for", example_row["sample_id"])
    for column in required_columns:
        print(" -", (base_dir / str(example_row[column])).resolve())
    spot_table = None
else:
    example_row = available_rows.iloc[0]
    spot_table = load_spot_table(
        sample_id=str(example_row["sample_id"]),
        features_path=(base_dir / str(example_row["features_path"])).resolve(),
        counts_path=(base_dir / str(example_row["counts_path"])).resolve(),
        coords_path=(base_dir / str(example_row["coords_path"])).resolve(),
        size_factor_path=(base_dir / str(example_row["size_factor_path"])).resolve(),
        min_total_counts=1.0,
    )
    print("Loaded", spot_table.sample_id)
    print("Features:", spot_table.features.shape)
    print("Counts:", spot_table.counts.shape)
    print("Coordinates:", None if spot_table.coords is None else spot_table.coords.shape)


## 4. Recompute the mean-one size factor

For valid spots in one slide, `SF = spot total / mean valid-spot total`.
This definition ensures a mean SF of one and supports the exact
decomposition `count = rate * SF`.


In [ ]:
if spot_table is not None:
    recomputed_sf, valid = compute_size_factor(spot_table.counts, min_total_counts=1.0)
    saved_sf = spot_table.size_factor
    print(f"Valid spots: {valid.sum():,} / {len(valid):,}")
    print(f"Mean recomputed SF: {recomputed_sf[valid].mean():.8f}")
    print(f"Maximum saved/recomputed difference: {np.max(np.abs(saved_sf[valid] - recomputed_sf[valid])):.3e}")
    assert np.isclose(recomputed_sf[valid].mean(), 1.0, atol=1e-6)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(log_size_factor(recomputed_sf[valid]), bins=50, color="#2878B5")
    axes[0].set_xlabel("log(SF)")
    axes[0].set_ylabel("Spots")
    axes[0].set_title("Within-slide SF distribution")
    if spot_table.coords is not None:
        points = axes[1].scatter(
            spot_table.coords[valid, 0],
            spot_table.coords[valid, 1],
            c=log_size_factor(recomputed_sf[valid]),
            s=5,
            cmap="magma",
            linewidths=0,
        )
        axes[1].invert_yaxis()
        axes[1].set_aspect("equal")
        axes[1].set_title("Measured log(SF)")
        fig.colorbar(points, ax=axes[1])
    plt.tight_layout()
    plt.show()


## 5. Prepared slide layout

A training-ready slide contains local HIPT features, 1,161-dimensional
context features, raw counts, coordinates, mean-one SF, spot identifiers
and gene identifiers. Training notebooks consume the committed manifest;
they do not infer splits from individual spots.

```text
data/HEST-1k/processed/<slide_id>/
  features.npy
  features_context.npy
  counts.npz
  coords.npy
  size_factor.npy
  spots.txt
  genes.txt
```
